# Week 5 Day 1 — Agent Foundations: Reasoning Loops, Tool Calling & Raw Python Agents

> **Goal:** Understand what an agent *actually* is under the hood by building one
> from scratch — no LangChain, no LangGraph, just the Anthropic API + a `while` loop.

**Setup:**
```bash
pip install anthropic python-dotenv
# Set your key in a .env file:  ANTHROPIC_API_KEY=sk-ant-...
```


## Setup & Imports

In [ ]:
import os, json, time, textwrap
import anthropic
from dotenv import load_dotenv
load_dotenv()

# Import our tool and agent modules
import sys
sys.path.insert(0, r'd:\Netixsol_Internship\netixsol_internship\Week-5\day-1')
from Tools import TOOL_SCHEMAS, execute_tool, TOOL_RUNNERS
from Agent  import RawPythonAgent, MODEL, MAX_ITERATIONS

print('anthropic version:', anthropic.__version__)
print('Tools available  :', list(TOOL_RUNNERS.keys()))
print('Model            :', MODEL)


---
## Task 1: Agent Concepts & Mental Model

### Agent vs Chatbot vs Workflow

| | Chatbot | Workflow | Agent |
|---|---|---|---|
| **Interaction** | Single-turn Q&A | Fixed sequence of steps | Dynamic, goal-directed |
| **Decision making** | None — prompt-in, text-out | Pre-coded branching | LLM decides what to do next |
| **Tool use** | None | Hard-coded API calls | Chooses tools autonomously |
| **Self-correction** | No | No | Yes — can retry or take different path |
| **Example** | GPT chat window | ETL pipeline | ReAct coding agent |

**What makes something *agentic*:**
1. **Autonomy** — decides its own next action without explicit human instruction per step
2. **Tool use** — can invoke external capabilities (APIs, code execution, file I/O)
3. **Multi-step planning** — decomposes a goal into a sequence of actions
4. **Self-correction** — observes results and adjusts behaviour (retry, reroute)

---

### The ReAct Pattern (Reason → Act → Observe → repeat)

```
User: "Which city is warmer, Karachi or London?"
         │
         ▼
┌─────────────────────────────────────────────────────────┐
│  WHILE  stop_reason != 'end_turn' (no more tool_use):   │
│                                                         │
│  1. REASON  ── LLM reads history, produces thought      │
│     "I need the weather in Karachi first..."            │
│                                                         │
│  2. ACT     ── LLM emits tool_use block                 │
│     {name: get_weather, input: {city: "Karachi"}}       │
│                                                         │
│  3. OBSERVE ── Python executes tool, appends result     │
│     tool_result: "34°C, Hot and sunny"                  │
│                                                         │
│     ──► loop back to step 1 ◄──                         │
│     LLM now looks up London, then calculates difference │
└─────────────────────────────────────────────────────────┘
         │
         ▼  stop_reason = 'end_turn'  (final text, no tool_use)
"Karachi (34°C) is 17°C warmer than London (17°C)."
```

---

### When an agent is overkill

> If you can answer the question with a **single prompt** or a **deterministic script**,
> use that instead — agents add latency, API cost, and failure surface.
> Use an agent only when the *path* to the answer is unknown in advance and requires
> dynamic decision-making across multiple steps.
>
> **Examples of overkill:** summarise this text, translate this sentence, classify
> this email, extract fields from a fixed-format document.
> A simple `client.messages.create()` call suffices for all of these.


---
## Task 2: Tool Calling Fundamentals

### Tool Schemas

Each tool has:
- `name` — identifier the model puts in `tool_use.name`
- `description` — **most important field**: Claude picks tools entirely from this
- `input_schema` — JSON Schema object the model must satisfy

**Why descriptions matter:**  
Claude sees *only* the name + description when deciding which tool to call.
A vague description like *"does math"* leads to missed calls or wrong tool selection.
A precise description stating inputs, outputs, units, and edge cases (e.g. division by zero)
leads to reliable, predictable calling behaviour.


In [ ]:
# Print all tool schemas in a readable format
for schema in TOOL_SCHEMAS:
    print(f"\n{'='*50}")
    print(f"Tool: {schema['name']}")
    print(f"Description: {schema['description'][:120]}...")
    props = schema['input_schema']['properties']
    print(f"Inputs: {list(props.keys())}")
    print(f"Required: {schema['input_schema'].get('required', [])}")


### Manual Tool Round-trip (no agent loop)


In [ ]:
# Demonstrate a single tool call manually (Task 2 requirement)
client = anthropic.Anthropic()

# Step 1: Send request — model chooses a tool
resp1 = client.messages.create(
    model='claude-3-5-haiku-20241022',
    max_tokens=512,
    tools=TOOL_SCHEMAS,
    messages=[{"role": "user", "content": "What is sqrt(225) + 15 squared?"}]
)
print('Stop reason:', resp1.stop_reason)
tool_block = next(b for b in resp1.content if b.type == 'tool_use')
print('Tool chosen:', tool_block.name)
print('Tool input :', tool_block.input)

# Step 2: Execute tool manually
result = execute_tool(tool_block.name, tool_block.input)
print('Tool result:', result)

# Step 3: Return tool_result and get final answer
resp2 = client.messages.create(
    model='claude-3-5-haiku-20241022',
    max_tokens=512,
    tools=TOOL_SCHEMAS,
    messages=[
        {"role": "user",      "content": "What is sqrt(225) + 15 squared?"},
        {"role": "assistant", "content": resp1.content},
        {"role": "user",      "content": [{
            "type": "tool_result",
            "tool_use_id": tool_block.id,
            "content": result
        }]}
    ]
)
final = next(b.text for b in resp2.content if b.type == 'text')
print('\nFinal answer:', final)


---
## Task 3: Minimal Agent Loop

The `RawPythonAgent` implements the full ReAct loop:
```python
while iteration < MAX_ITERATIONS:
    response = call_llm(conversation_history)
    if response.stop_reason == 'end_turn' and no tool_use:
        return final_text   # done
    for tool_call in extract_tool_uses(response):
        result = execute_tool(tool_call.name, tool_call.input)
        append tool_result to conversation_history
```


In [ ]:
agent = RawPythonAgent(max_iterations=10)


### Test 1: Single tool call


In [ ]:
answer = agent.run_fresh(
    'What is the square root of 256 plus 7 to the power of 3?'
)


### Test 2: Multi-step — weather comparison (2 tool calls)


In [ ]:
answer = agent.run_fresh(
    'Look up the current weather in Karachi and London. '
    'Which city is warmer, and by how many degrees Celsius?'
)


### Test 3: Three tool calls — weather + unit conversions


In [ ]:
answer = agent.run_fresh(
    'What is the temperature in Dubai? '
    'Convert that temperature to both Fahrenheit and Kelvin.'
)


### Test 4: Complex — 2 weather lookups + average calculation


In [ ]:
answer = agent.run_fresh(
    'Find the temperatures in Tokyo and Sydney. '
    'Calculate their average and tell me which is warmer.'
)


---
## Task 4: Memory & State Handling

### Two kinds of memory in this agent

| Memory type | What it is | Where it lives | Who reads it |
|---|---|---|---|
| **Conversation memory** | The full `messages` list passed to the API on every turn | `agent.conversation_history` | The LLM (full context window) |
| **Working memory** | Scratchpad: iteration count, tools called, errors seen, start time | `agent.working_state` | The Python agent code (for logging/guardrails) |

Conversation memory is how the agent *knows what it already did*.
Working memory is how the *developer* monitors and controls the agent.

### Multi-turn chat (history persists)


In [ ]:
# Multi-turn: history persists between calls
agent.clear_history()
agent.chat('What is the weather in Karachi?')
agent.chat('How does that compare to London?')  # agent remembers Karachi answer

print('\nWorking state after multi-turn:')
for k, v in agent.working_state.items():
    print(f'  {k}: {v}')

print('\nConversation history length:', len(agent.conversation_history), 'messages')


---
## Task 5: Failure Modes & Guardrails

### Deliberately Breaking the Agent


#### Failure 1: Ambiguous request (no city specified)


In [ ]:
# Model will have to ask for clarification or guess — observe behaviour
agent.run_fresh("What's the weather like?")


#### Failure 2: Tool returns an error (city not in database)


In [ ]:
agent.run_fresh('What is the weather in Narnia?')


#### Failure 3: Task needs an undefined tool (web search)


In [ ]:
# Model has no search tool — observe graceful degradation
agent.run_fresh(
    'Search the web for the 3 most recent AI safety papers and summarise them.'
)


#### Failure 4: Bad/malicious tool argument


In [ ]:
# Calculator's allowlist blocks this — observe safe error handling
agent.run_fresh("Calculate the result of 'import os; os.system(\"whoami\")'")


---
### Failure Mode Catalogue

| # | Failure Mode | What happens | Mitigation |
|---|---|---|---|
| 1 | **Infinite loop** | Model keeps calling tools, never reaches `end_turn` | `MAX_ITERATIONS` safeguard + timeout |
| 2 | **Hallucinated tool call** | Model invents a tool name not in the schema | `execute_tool` returns `ERROR: tool not defined`; model is told via `tool_result` |
| 3 | **Wrong tool arguments** | Model passes wrong types or missing required fields | JSON Schema validation before calling runner; return structured error |
| 4 | **Silent tool error** | Tool crashes but exception is swallowed | Wrap all runners in `try/except`; return `ERROR:` string; set `is_error=True` |
| 5 | **Ambiguous request** | Model doesn't have enough context to act | Model asks clarifying question (end_turn without tool_use) — good behaviour |
| 6 | **Missing tool** | User asks for capability we haven't defined | Model either asks user to provide info manually, or answers from training data |

---

### Why frameworks like LangChain / LangGraph / CrewAI exist

> We just built a working agent in ~150 lines of Python, but every production
> team re-discovers the same problems:
> retries, streaming, token counting, state persistence, human-in-the-loop
> pauses, parallel tool calls, multi-agent routing, observability, prompt versioning.
> Frameworks package these battle-tested patterns so engineers don't reinvent them.
> The tradeoff is opacity — knowing what we built today means you can debug
> *inside* the framework when it behaves unexpectedly, rather than treating it
> as a black box.
